# 4. Phylogeny
Since we are using 16S rRNA, we will use a refernce-based method to build our phylogenetic tree. In particular we will use fragment insertion which places short query sequences (e.g., ASVs) into a high-quality existing reference tree using SATé-enabled phylogenetic placement (SEPP). Our reference tree is built from the SILVA 13_8 database.
SILVA Reference alignments are particularly powerful for rRNA gene sequence data, as knowledge of secondary structure is incorporated into the curation process, thus increasing alignment quality.
This will give us accurate UniFrac distances without reconstructing everything from scratch.

In [1]:
# 1- Import packages
import os
import pandas as pd
from qiime2 import Visualization
import matplotlib.pyplot as plt
import numpy as np
import qiime2 as q2
%matplotlib inline

In [2]:
# 2 - Set working directory
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/Project/MicrobiomeAnalysis_TummyTribe/scripts").


In [3]:
# 3 - Data directories
raw_data_dir = "../data/raw"
phylogeny_data_dir = "../data/processed/phylogeny"
denoising_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"
metadata_dir = "../data/raw"

## Fragment insertion

We can get the SILVA 128 reference tree from qiime. According to a user in the qiime forum using SILVA 138 for the taxonomy and SILVA 128 to build the phylogenetic tree should not lead to any complications (https://forum.qiime2.org/t/compatibility-of-sepp-silva128-with-taxonomy-classification-using-silva138/31172?utm_source=chatgpt.com).

In [18]:
#SILVA 128 for SEPP
! wget -O $phylogeny_data_dir/silva-128-sepp-refs.qza https://data.qiime2.org/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza

--2025-11-11 13:28:36--  https://data.qiime2.org/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza
Resolving data.qiime2.org (data.qiime2.org)... 54.200.1.12
Connecting to data.qiime2.org (data.qiime2.org)|54.200.1.12|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza [following]
--2025-11-11 13:28:36--  https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.218.246.176, 52.92.162.192, 52.218.236.160, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.218.246.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 181253322 (173M) [binary/octet-stream]
Saving to: ‘../data/processed/phylogeny/silva-128-sepp-refs.qza’

../data/processed/p 100%[===================>] 172.86M  16.3MB/s    in 12s     

2025-11-11

In [19]:
!qiime tools peek $denoising_data_dir/dada2_rep_seq.qza 

UUID:        0224622c-56b2-43d5-8034-c3d398690a51
Type:        FeatureData[Sequence]
Data format: DNASequencesDirectoryFormat


In [22]:
!qiime tools peek $phylogeny_data_dir/silva-128-sepp-refs.qza

UUID:        e44b5e78-31e5-4a0f-9041-494bc3ca2df2
Type:        SeppReferenceDatabase
Data format: SeppReferenceDirFmt


In [21]:
#does not run in notebook -> euler (needs amplicon env)
! qiime fragment-insertion sepp \
    --i-representative-sequences $denoising_data_dir/dada2_rep_seq.qza \
    --i-reference-database $phylogeny_data_dir/silva-128-sepp-refs.qza \
    --p-threads 2 \
    --o-tree $phylogeny_data_dir/sepp-tree.qza \
    --o-placements $phylogeny_data_dir/sepp-tree-placements.qza \
    --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Removing /tmp/tmp.1ZZbpa3ZS5/sepp-tmp-QfUBleakIt


In [4]:
! qiime empress community-plot \
    --i-tree $phylogeny_data_dir/sepp-tree.qza \
    --i-feature-table $denoising_data_dir/dada2_table.qza \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --m-sample-metadata-file $metadata_dir/metadata.tsv \
    --o-visualization $phylogeny_data_dir/empress-sepp-tree.qzv

Error: QIIME 2 has no plugin/command named 'empress'.


In [6]:
! qiime phylogeny view-tree \
    --i-tree sepp-tree.qza \
    --o-visualization sepp-tree.qzv


Error: QIIME 2 plugin 'phylogeny' has no action 'view-tree'.


In [ ]:
! qiime tools view $phylogeny_data_dir/empress-sepp-tree.qzv

In [ ]:
#does this change anything????